<a href="https://colab.research.google.com/github/Rachelllle/Aircraft-Analysis-Engine/blob/load_vectorize_images_corrected/aircraft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("clé Kaggle  installée")

In [ ]:
!kaggle datasets download -d seryouxblaster764/fgvc-aircraft

Dataset URL: https://www.kaggle.com/datasets/seryouxblaster764/fgvc-aircraft
License(s): other
fgvc-aircraft.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!mkdir -p aircraft_data

In [ ]:
!unzip -o -q fgvc-aircraft.zip -d ./aircraft_data

In [ ]:
!ls -R ./aircraft_data | head -n 30

./aircraft_data:
fgvc-aircraft-2013b
model_manufacturer
ms_embeddings.parquet
ms_parsed_full.parquet
test.csv
train.csv
val.csv

./aircraft_data/fgvc-aircraft-2013b:
fgvc-aircraft-2013b

./aircraft_data/fgvc-aircraft-2013b/fgvc-aircraft-2013b:
data
evaluation.m
example_evaluation.m
README.html
README.md
vl_argparse.m
vl_pr.m
vl_roc.m
vl_tpfp.m

./aircraft_data/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data:
families.txt
images
images_box.txt
images_family_test.txt
images_family_train.txt
images_family_trainval.txt


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, element_at
from pyspark.sql.functions import lit, regexp_replace

spark = SparkSession.builder \
    .appName("aircraft_MS1_parsing") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

In [ ]:
"""data_path = "./aircraft_data/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data/"
raw_manuf = spark.read.text(data_path + "images_manufacturer_train.txt")

df_labels = raw_manuf.select(
    split(col("value"), " ", 2).getItem(0).alias("image_id"),
    split(col("value"), " ", 2).getItem(1).alias("manufacturer")
)


df_images_raw = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.jpg") \
    .load(data_path + "images/")

df_images = df_images_raw.withColumn(
    "image_id",
    split(element_at(split(col("path"), "/"), -1), r"\.").getItem(0)
)

df_ms1_output = df_images.join(df_labels, "image_id") \
    .select("image_id", "manufacturer", "content")"""


data_path = "./aircraft_data/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data/"

def read_annotation_file(filepath, label_name):
    df = spark.read.text(filepath)
    df = df.select(
        split(col("value"), " ", 2).getItem(0).alias("image_id"),
        split(col("value"), " ", 2).getItem(1).alias(label_name)
    )
    return df

def build_annotation_df(split_name):
    df_manuf   = read_annotation_file(data_path + f"images_manufacturer_{split_name}.txt", "manufacturer")
    df_family  = read_annotation_file(data_path + f"images_family_{split_name}.txt",       "family")
    df_variant = read_annotation_file(data_path + f"images_variant_{split_name}.txt",      "variant")

    df = df_manuf.join(df_family, "image_id").join(df_variant, "image_id")
    df = df.withColumn("split", lit(split_name))
    return df

df_all_annotations = build_annotation_df("train") \
    .union(build_annotation_df("val")) \
    .union(build_annotation_df("test"))

df_all_annotations.show(5)
print(f"Total: {df_all_annotations.count()} images")

<>:16: SyntaxWarning: invalid escape sequence '\.'
<>:16: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-28935/759766677.py:16: SyntaxWarning: invalid escape sequence '\.'
  split(element_at(split(col("path"), "/"), -1), r"\.").getItem(0)


+--------+------------+----------+-------+-----+
|image_id|manufacturer|    family|variant|split|
+--------+------------+----------+-------+-----+
| 1025794|      Boeing|Boeing 707|707-320|train|
| 1340192|      Boeing|Boeing 707|707-320|train|
| 0056978|      Boeing|Boeing 707|707-320|train|
| 0698580|      Boeing|Boeing 707|707-320|train|
| 0450014|      Boeing|Boeing 707|707-320|train|
+--------+------------+----------+-------+-----+
only showing top 5 rows
Total: 10000 images


In [ ]:
#added my rachel
from pyspark.sql.functions import regexp_replace

csv_path = "./aircraft_data/"

df_csv = spark.read.csv(csv_path + "train.csv", header=True).withColumn("split", lit("train")) \
    .union(spark.read.csv(csv_path + "val.csv",  header=True).withColumn("split", lit("val"))) \
    .union(spark.read.csv(csv_path + "test.csv", header=True).withColumn("split", lit("test")))

df_csv = df_csv.withColumn("image_id", regexp_replace(col("filename"), "\\.jpg$", ""))

df_csv.show(5)
print(f"Total CSV: {df_csv.count()} lignes")

+-----------+-------+------+-----+--------+
|   filename|Classes|Labels|split|image_id|
+-----------+-------+------+-----+--------+
|1025794.jpg|707-320|     0|train| 1025794|
|1340192.jpg|707-320|     0|train| 1340192|
|0056978.jpg|707-320|     0|train| 0056978|
|0698580.jpg|707-320|     0|train| 0698580|
|0450014.jpg|707-320|     0|train| 0450014|
+-----------+-------+------+-----+--------+
only showing top 5 rows
Total CSV: 10000 lignes


In [ ]:
#added by rachel
df_images = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.jpg") \
    .load(data_path + "images/") \
    .withColumn("image_id", split(element_at(split(col("path"), "/"), -1), "\\.").getItem(0)) \
    .select("image_id", "content")

df_parsed = df_images \
    .join(df_all_annotations, "image_id") \
    .join(df_csv.select("image_id", "Classes", "Labels"), "image_id") \
    .withColumnRenamed("Classes", "variant_class") \
    .withColumnRenamed("Labels",  "variant_label")

df_parsed.printSchema()
df_parsed.drop("content").write.mode("overwrite").parquet("./aircraft_data/ms_parsed_full.parquet")
print("Parsing complet sauvegardé!")

root
 |-- image_id: string (nullable = true)
 |-- content: binary (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- family: string (nullable = true)
 |-- variant: string (nullable = true)
 |-- split: string (nullable = false)
 |-- variant_class: string (nullable = true)
 |-- variant_label: string (nullable = true)

✅ Parsing complet sauvegardé!


In [ ]:
"""df_ms1_output.write.mode("overwrite").parquet("./aircraft_data/ms1_output.parquet")
print("Data parsées  stockées dans ms1_output.parquet")"""

'df_ms1_output.write.mode("overwrite").parquet("./aircraft_data/ms1_output.parquet")\nprint("Data parsées  stockées dans ms1_output.parquet")'

In [ ]:
import io
import numpy as np
import pandas as pd
from PIL import Image
from pyspark.sql.functions import pandas_udf
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf

In [ ]:
#added by rache
IMG_SIZE = 64  # 64x64x3 = 12,288 features (random forest)

@pandas_udf("array<float>")
def pixels_udf(content_series: pd.Series) -> pd.Series:
    def process(content):
        try:
            img = Image.open(io.BytesIO(bytes(content))).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            arr = np.array(img, dtype=np.float32).flatten() / 255.0
            return arr.tolist()
        except:
            return [0.0] * (IMG_SIZE * IMG_SIZE * 3)
    return content_series.apply(process)

print(f"UDF prêt — {IMG_SIZE}x{IMG_SIZE}x3 = {IMG_SIZE*IMG_SIZE*3} features par image")

✅ UDF prêt — 64x64x3 = 12288 features par image


In [ ]:
list_to_vector_udf = udf(lambda l: Vectors.dense(l), VectorUDT())

df_emb = df_parsed.repartition(20) \
    .withColumn("pixels_raw", pixels_udf(col("content"))) \
    .withColumn("features", list_to_vector_udf(col("pixels_raw"))) \
    .drop("content", "pixels_raw")

df_emb.write.mode("overwrite").parquet("./aircraft_data/ms_embeddings.parquet")
print("Features sauvegardées!")
df_emb.printSchema()

✅ Features sauvegardées!
root
 |-- image_id: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- family: string (nullable = true)
 |-- variant: string (nullable = true)
 |-- split: string (nullable = false)
 |-- variant_class: string (nullable = true)
 |-- variant_label: string (nullable = true)
 |-- features: vector (nullable = true)



In [ ]:
#added by rache
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

df_emb = spark.read.parquet("./aircraft_data/ms_embeddings.parquet")

df_train = df_emb.filter(col("split") == "train")
df_test  = df_emb.filter(col("split") == "test")

print(f"Train: {df_train.count()} | Test: {df_test.count()}")

Train: 3334 | Test: 3333


In [ ]:
#added by rache
def train_model(df_train, target_col, model_name, max_depth=8):
    indexer = StringIndexer(inputCol=target_col, outputCol="label", handleInvalid="skip")
    rf = RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=50, maxDepth=max_depth, maxBins=32, seed=42
    )
    pipeline = Pipeline(stages=[indexer, rf])
    print(f"Training {model_name}...")
    model = pipeline.fit(df_train)
    model.write().overwrite().save(f"./aircraft_data/model_{model_name}")
    print(f"{model_name} done!")
    return model

model_manuf   = train_model(df_train, "manufacturer", "manufacturer", max_depth=8)
model_family  = train_model(df_train, "family",       "family",       max_depth=8)
model_variant = train_model(df_train, "variant",      "variant",      max_depth=8)

⏳ Training manufacturer...
✅ manufacturer done!
⏳ Training family...
✅ family done!
⏳ Training variant...
✅ variant done!


In [ ]:
"""IMG_SIZE = 224

@pandas_udf("array<float>")
def resize_image_udf(content_series: pd.Series) -> pd.Series:
    def resize(content):
        img = Image.open(io.BytesIO(content)).convert('RGB')
        img = img.resize((IMG_SIZE, IMG_SIZE))
        return np.array(img).flatten().astype(float) / 255.0
    return content_series.apply(resize)

df_ready = spark.read.parquet("./aircraft_data/ms1_output.parquet") \
    .withColumn("features_raw", resize_image_udf(col("content")))"""

'IMG_SIZE = 224\n\n@pandas_udf("array<float>")\ndef resize_image_udf(content_series: pd.Series) -> pd.Series:\n    def resize(content):\n        img = Image.open(io.BytesIO(content)).convert(\'RGB\')\n        img = img.resize((IMG_SIZE, IMG_SIZE))\n        return np.array(img).flatten().astype(float) / 255.0\n    return content_series.apply(resize)\n\ndf_ready = spark.read.parquet("./aircraft_data/ms1_output.parquet")     .withColumn("features_raw", resize_image_udf(col("content")))'

In [ ]:

"""from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import udf"""

'from pyspark.ml.linalg import Vectors, VectorUDT\nfrom pyspark.ml.feature import StringIndexer\nfrom pyspark.sql.functions import udf'

In [ ]:
"""#Convertir l'array de pixels en Vecteur Dense Spark pour que spark lit les images
list_to_vector_udf = udf(lambda l: Vectors.dense(l), VectorUDT())

df_ml = df_ready.withColumn("features", list_to_vector_udf(col("features_raw")))

indexer = StringIndexer(inputCol="manufacturer", outputCol="label")
indexer_model = indexer.fit(df_ml)
df_final_ml = indexer_model.transform(df_ml)

train_data, test_data = df_final_ml.randomSplit([0.7, 0.3], seed=42) #3334 images train / 3333 test"""

'#Convertir l\'array de pixels en Vecteur Dense Spark pour que spark lit les images\nlist_to_vector_udf = udf(lambda l: Vectors.dense(l), VectorUDT())\n\ndf_ml = df_ready.withColumn("features", list_to_vector_udf(col("features_raw")))\n\nindexer = StringIndexer(inputCol="manufacturer", outputCol="label")\nindexer_model = indexer.fit(df_ml)\ndf_final_ml = indexer_model.transform(df_ml)\n\ntrain_data, test_data = df_final_ml.randomSplit([0.7, 0.3], seed=42) #3334 images train / 3333 test'

In [ ]:
"""import io
from PIL import Image

df_viz = spark.read.parquet("./aircraft_data/ms1_output.parquet").limit(5).toPandas()

plt.figure(figsize=(15, 5))

for i, row in df_viz.iterrows():
    img = Image.open(io.BytesIO(row['content']))

    plt.subplot(1, 5, i+1)
    plt.imshow(img)
    plt.title(row['manufacturer'])
    plt.axis('off')

plt.show()"""

'import io\nfrom PIL import Image\n\ndf_viz = spark.read.parquet("./aircraft_data/ms1_output.parquet").limit(5).toPandas()\n\nplt.figure(figsize=(15, 5))\n\nfor i, row in df_viz.iterrows():\n    img = Image.open(io.BytesIO(row[\'content\']))\n\n    plt.subplot(1, 5, i+1)\n    plt.imshow(img)\n    plt.title(row[\'manufacturer\'])\n    plt.axis(\'off\')\n\nplt.show()'